# kaiming-uniform-init — ex2: Kaiming-uniform Conv2d init in fan_in vs fan_out mode

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kaiming-uniform-init`. Running the final beacon cell reports progress against the `Init: Kaiming uniform` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-init`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-init"
DD_SUBTOPIC = "Init: Kaiming uniform"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kaiming uniform — fan_in vs fan_out

Ex1 initialized a Linear weight with bound `1/sqrt(fan_in)`. The deepening move is `fan_out` mode and a CONVOLUTIONAL weight where fan_in ≠ fan_out.

For a `Conv2d(in_ch, out_ch, kernel=k)` weight of shape `(out_ch, in_ch, k, k)`:
- `fan_in  = in_ch  * k * k` — receptive field × input channels
- `fan_out = out_ch * k * k` — receptive field × output channels

```python
# Kaiming-uniform bound = gain * sqrt(3 / fan)
# For relu nonlinearity, gain = sqrt(2).
# So bound = sqrt(2) * sqrt(3 / fan) = sqrt(6 / fan).
bound_in  = math.sqrt(6.0 / fan_in)
bound_out = math.sqrt(6.0 / fan_out)
```

**Why two modes exist.** `fan_in` preserves the variance of activations on the FORWARD pass; `fan_out` preserves it on the BACKWARD pass. For a Conv expanding 3→64 channels, `fan_in=27` and `fan_out=576` produce DIFFERENT bounds — `fan_out` mode shrinks weights ~4.6× more.

**Default in PyTorch.** `nn.init.kaiming_uniform_` defaults to `mode='fan_in', nonlinearity='leaky_relu', a=sqrt(5)` for legacy compatibility. Most modern code passes `nonlinearity='relu'` explicitly.

### Exercise 2 — Kaiming-uniform Conv2d init in fan_in vs fan_out mode

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Kaiming-uniform bound `sqrt(6/fan)` to a Conv2d weight in BOTH `'fan_in'` and `'fan_out'` mode, returning the two initialized weight tensors and their empirical max-abs values for comparison.
> Keywords: kaiming, conv2d, fan_in, fan_out, init
> ```

**KCs targeted:** `conv-weight-fan-in-vs-fan-out-formula`, `kaiming-uniform-relu-bound`

Implement `ex2_kaiming_conv_two_modes(in_ch, out_ch, kernel)`. Initialize a Conv2d weight in BOTH modes and report the bounds.

Use `nn.init.kaiming_uniform_(weight, mode=..., nonlinearity='relu')`.

Return a dict with EXACTLY these keys:

- `'fan_in'`: `int`, `in_ch * kernel * kernel`.
- `'fan_out'`: `int`, `out_ch * kernel * kernel`.
- `'bound_fan_in'`: `float`, `sqrt(6.0 / fan_in)` (the theoretical uniform bound for `nonlinearity='relu'`, i.e. gain=sqrt(2)).
- `'bound_fan_out'`: `float`, `sqrt(6.0 / fan_out)`.
- `'weight_fan_in'`: `torch.Tensor`, the Conv2d weight of shape `(out_ch, in_ch, kernel, kernel)` initialized with `mode='fan_in'`.
- `'weight_fan_out'`: `torch.Tensor`, same shape, `mode='fan_out'`.
- `'empirical_max_in'`: `float`, `weight_fan_in.abs().max().item()`.
- `'empirical_max_out'`: `float`, `weight_fan_out.abs().max().item()`.

Constraints:
- Build each weight as an empty `(out_ch, in_ch, kernel, kernel)` tensor — do NOT construct a full `nn.Conv2d` module.
- Seed `t.manual_seed(0)` BEFORE the first init and AGAIN before the second so the two are directly comparable.
- The empirical max-abs MUST be `<= bound` (uniform is bounded).

In [ ]:
def ex2_kaiming_conv_two_modes(in_ch, out_ch, kernel):
    import math
    fan_in = in_ch * kernel * kernel
    fan_out = out_ch * kernel * kernel
    bound_in = math.sqrt(6.0 / fan_in)
    bound_out = math.sqrt(6.0 / fan_out)
    shape = (out_ch, in_ch, kernel, kernel)

    t.manual_seed(0)
    w_in = t.empty(*shape)
    t.nn.init.kaiming_uniform_(w_in, mode='fan_in', nonlinearity='relu')

    t.manual_seed(0)
    w_out = t.empty(*shape)
    t.nn.init.kaiming_uniform_(w_out, mode='fan_out', nonlinearity='relu')

    return {
        'fan_in': fan_in,
        'fan_out': fan_out,
        'bound_fan_in': bound_in,
        'bound_fan_out': bound_out,
        'weight_fan_in': w_in,
        'weight_fan_out': w_out,
        'empirical_max_in': w_in.abs().max().item(),
        'empirical_max_out': w_out.abs().max().item(),
    }


<details><summary>Solution</summary>

```python
def ex2_kaiming_conv_two_modes(in_ch, out_ch, kernel):
    import math
    fan_in = in_ch * kernel * kernel
    fan_out = out_ch * kernel * kernel
    bound_in = math.sqrt(6.0 / fan_in)
    bound_out = math.sqrt(6.0 / fan_out)
    shape = (out_ch, in_ch, kernel, kernel)

    t.manual_seed(0)
    w_in = t.empty(*shape)
    t.nn.init.kaiming_uniform_(w_in, mode='fan_in', nonlinearity='relu')

    t.manual_seed(0)
    w_out = t.empty(*shape)
    t.nn.init.kaiming_uniform_(w_out, mode='fan_out', nonlinearity='relu')

    return {
        'fan_in': fan_in,
        'fan_out': fan_out,
        'bound_fan_in': bound_in,
        'bound_fan_out': bound_out,
        'weight_fan_in': w_in,
        'weight_fan_out': w_out,
        'empirical_max_in': w_in.abs().max().item(),
        'empirical_max_out': w_out.abs().max().item(),
    }
```

**`nonlinearity='relu'` ⇒ gain=sqrt(2).** PyTorch's gain table (`torch.nn.init.calculate_gain`) gives sqrt(2) for relu. The uniform bound is `gain * sqrt(3/fan)` = `sqrt(2) * sqrt(3/fan)` = `sqrt(6/fan)`. Pass `nonlinearity='relu'` explicitly; the default `'leaky_relu'` with `a=sqrt(5)` is the legacy behavior for backward compatibility with old PyTorch defaults.

**Same seed for both modes.** Without resetting the seed, the fan_out init would consume different random numbers than fan_in, and the comparison would be confounded by the RNG state. `t.manual_seed(0)` before each init guarantees the only difference is the SCALE (the bound), not the underlying samples.

**Why expanding convs use fan_in mode by default.** Forward-pass variance preservation. For a Conv 3→64, fan_in=27 weights contribute to each output activation; the variance of that sum needs the weight variance scaled by `2/fan_in`. fan_out preserves the BACKWARD pass instead — useful when the bottleneck is gradient flow, not forward signal.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()